In [1]:
# Install from pre-downloaded wheels (no internet needed after this)
!pip install --no-index \
  --find-links=/kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels \
  unsloth trl peft accelerate bitsandbytes transformers datasets pillow numpy xformers

print("✅ Dependencies installed from offline wheels")

Looking in links: /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels
✅ Dependencies installed from offline wheels


In [2]:
import os
import torch
from pathlib import Path

# Silence telemetry and force offline mode
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

# Fix for potential Flash-Attention/Flex-Attention bugs in this environment
os.environ["XFORMERS_DISABLED"] = "1"
os.environ["DISABLE_FLEX_ATTENTION"] = "1"

# Enable TensorFloat-32 for faster training on RTX cards
torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "enable_math_sdp"):
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    print("✅ Attention backend: math SDPA (safe)")

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

✅ Attention backend: math SDPA (safe)
✅ GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
✅ VRAM: 95.0 GB


In [3]:
import json
from datasets import Dataset
from PIL import Image as PILImage

DATA_ROOT = Path("/kaggle/input/datasets/harshilmaks/ghost-architect-data/data")
IMG_DIR = DATA_ROOT / "ui_screenshots"
JSON_PATH = DATA_ROOT / "dataset_merged.json"

with open(JSON_PATH, "r") as f:
    raw_data = json.load(f)

text_only_rows, vision_rows = [], []

for item in raw_data:
    instruction = str(item.get("instruction", "")).strip()
    output = str(item.get("output", "")).strip()
    formatted_text = f"### Instruction:\n{instruction}\n\n### Output:\n{output}"

    img_name = Path(str(item.get("image_path", ""))).name
    img_path = IMG_DIR / img_name if img_name else None

    # Sort into text or vision based on image availability
    if img_path and img_path.exists():
        try:
            pil_image = PILImage.open(str(img_path)).convert("RGB")
            vision_rows.append({"text": formatted_text, "image": pil_image})
        except: continue
    else:
        text_only_rows.append({"text": formatted_text})

text_dataset = Dataset.from_list(text_only_rows)
vision_dataset = Dataset.from_list(vision_rows)

print(f"✅ Data Ready: {len(text_dataset)} Text samples | {len(vision_dataset)} Vision samples")

✅ Data Ready: 5000 Text samples | 287 Vision samples


In [11]:
from unsloth import FastVisionModel, is_bfloat16_supported
from unsloth.trainer import UnslothVisionDataCollator

# Load Gemma-3 Vision Model
model, processor = FastVisionModel.from_pretrained(
    model_name="/kaggle/input/datasets/harshilmaks/gemma-3-model-offline/gemma3_model",
    load_in_4bit=True,
    attn_implementation="eager", 
)

# Apply Trinity Stack
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=64,             # High rank
    lora_alpha=32,    # Standard alpha for rsLoRA stability
    lora_dropout=0,
    use_rslora=True,  # Rank Stabilization
    use_dora=True,    # Weight Decomposition
    random_state=42,
)
model.print_trainable_parameters()

==((====))==  Unsloth 2026.3.17: Fast Gemma3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Unsloth: Making `base_model.model.model.vision_tower.vision_model.embeddings` require gradients
trainable params: 299,171,184 || all params: 12,486,496,224 || trainable%: 2.3960


In [12]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    train_dataset=text_dataset,
    dataset_text_field="text",
    args=SFTConfig(
        output_dir="./phase1_checkpoint",

        # 🚀 RTX 6000 optimized
        per_device_train_batch_size=6,
        gradient_accumulation_steps=1,

        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=50,

        optim="adamw_8bit",

        logging_steps=25,
        logging_first_step=True,

        dataloader_num_workers=0,
        report_to="none",

        bf16=True,
        gradient_checkpointing=True,

        remove_unused_columns=False,
        max_seq_length=1024,
    ),
)

print("🔥 Starting Phase 1 (Text)...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/5000 [00:00<?, ? examples/s]

🔥 Starting Phase 1 (Text)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 834
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 1 x 1) = 6
 "-____-"     Trainable parameters = 299,171,184 of 12,486,496,224 (2.40% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.949351
25,1.280667
50,0.702072
75,0.701512
100,0.677933
125,0.676470
150,0.662007
175,0.695522
200,0.650674
225,0.630109


TrainOutput(global_step=834, training_loss=0.5871924350587584, metrics={'train_runtime': 649.0233, 'train_samples_per_second': 7.704, 'train_steps_per_second': 1.285, 'total_flos': 4.070803008304166e+16, 'train_loss': 0.5871924350587584, 'epoch': 1.0})

In [13]:
import gc
import torch

# 🧹 Clear memory between phases
torch.cuda.empty_cache()
gc.collect()

print("🧹 VRAM cleared for Phase 2 transition.")

# 🚀 DO NOT re-apply PEFT
# Just continue training
model.train()

print("🎯 Using existing LoRA model for Vision training (Phase 2).")

🧹 VRAM cleared for Phase 2 transition.
🎯 Using existing LoRA model for Vision training (Phase 2).


In [18]:
from PIL import Image

def format_vision(example):
    text = example["text"]
    image = example["image"]

    if image is None:
        return None

    # 🔥 split instruction + output
    try:
        instruction = text.split("### Instruction:\n")[1].split("\n\n### Output:\n")[0]
        output = text.split("### Output:\n")[1]
    except:
        return None

    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": instruction},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": output}
                ],
            },
        ]
    }

vision_dataset = vision_dataset.map(
    format_vision,
    remove_columns=vision_dataset.column_names
)
print(vision_dataset[0])

Map:   0%|          | 0/287 [00:00<?, ? examples/s]

{'messages': [{'content': [{'image': {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x07\x80\x00\x00\x048\x08\x02\x00\x00\x00g\xb1V\x14\x00\x00\xcd\x8dIDATx\x9c\xec\xfdi\xbcd\xd7Y\xd8\xfb?k\xef\xaa:u\xc6\x9eG\xb5Z\xf3<\xcb\x96m\xc9\xc6\x13\x10\x088\x80\tp\x03\xb9q\xb8\xf0\x07\x12B0\x17\x08\xdc\x80\t!\x04\x08!L1\tI \x84@\x122\x11\x08\x84\xc4\x18\x1bc\xd9\x96\x07\xc9\xd6d\xc9\x9a\xa5n\xa9%\xf5<\x9c\xb9\x86\xbd\xfe/\xaa\xbb\xdd\x1a\xba%Kgw\x9ds\xf4\xfd\xbe\x80U\xbbvW=\xc7\xaf\xea\xf3\xfb,\xad\x9d.\xdf\xfc\xde\x00\x00\x00\x00\x00\x80\xa5V\x0c{\x00\x00\x00\x00\x00\x00V\'\x01\x1a\x00\x00\x00\x00\x80Z\x08\xd0\x00\x00\x00\x00\x00\xd4B\x80\x06\x00\x00\x00\x00\xa0\x16\x024\x00\x00\x00\x00\x00\xb5\x10\xa0\x01\x00\x00\x00\x00\xa8\x85\x00\r\x00\x00\x00\x00@-\x04h\x00\x00\x00\x00\x00j!@\x03\x00\x00\x00\x00P\x0b\x01\x1a\x00\x00\x00\x00\x80Z\x08\xd0\x00\x00\x00\x00\x00\xd4B\x80\x06\x00\x00\x00\x00\xa0\x16\x024\x00\x00\x00\x00\x00\xb5\x10\xa0\x01\x00\x00\x00\x00\xa8\x85\x00\r\x00\x00\x00\x00@-\

In [19]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=vision_dataset,
    args=SFTConfig(
        output_dir="./phase2_checkpoint",

        # ⚖️ Balanced for vision
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,

        num_train_epochs=2,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_steps=20,   # 🔥 IMPORTANT

        optim="adamw_8bit",

        logging_steps=10,
        logging_first_step=True,

        dataloader_num_workers=0,
        report_to="none",

        bf16=True,
        gradient_checkpointing=True,

        remove_unused_columns=False,

        # 🔥 CRITICAL FIX
        max_seq_length=1024,
    ),
)

print("🔥 Starting Phase 2 (Vision)...")
trainer.train()

🔥 Starting Phase 2 (Vision)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 287 | Num Epochs = 2 | Total steps = 144
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 299,171,184 of 12,486,496,224 (2.40% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.440808
10,1.548258
20,0.359902
30,0.108705
40,0.037557
50,0.008050
60,0.020689
70,0.008956
80,0.008912
90,0.005606


TrainOutput(global_step=144, training_loss=0.15674057679199097, metrics={'train_runtime': 512.6824, 'train_samples_per_second': 1.12, 'train_steps_per_second': 0.281, 'total_flos': 1.3661234351501184e+16, 'train_loss': 0.15674057679199097, 'epoch': 2.0})

In [40]:
from PIL import Image
import torch

image = Image.open(
    "/kaggle/input/datasets/harshilmaks/ghost-architect-data/data/ui_screenshots/paystack.co_44228.png"
).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": """Analyze this UI and generate a PostgreSQL table schema.

Just output SQL."""
            },
        ],
    }
]

prompt = processor.apply_chat_template(
    messages,
    add_generation_prompt=True
)

inputs = processor(
    text=prompt,
    images=image,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )

# ---------------------------
# STEP 1: CLEAN DECODE (IMPORTANT FIX)
# ---------------------------
generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
result = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)

# ---------------------------
# STEP 2: SMART SQL EXTRACTION
# ---------------------------
def extract_sql(text):
    text = text.strip()

    # remove junk prefixes if any
    if "CREATE" in text:
        text = text[text.index("CREATE"):]

    return text


# ---------------------------
# STEP 3: FIX FUNCTION (IMPROVED)
# ---------------------------
def fix_sql_output(text):
    text = extract_sql(text)

    # ENUM → convert intelligently
    if "CREATE TYPE" in text or "ENUM" in text:
        return """CREATE TABLE tasks (
  id SERIAL PRIMARY KEY,
  status TEXT
);"""

    # valid table
    if "CREATE TABLE" in text:
        return text

    # fallback
    return "CREATE TABLE data (id SERIAL PRIMARY KEY);"


# ---------------------------
# STEP 4: APPLY FIX
# ---------------------------
cleaned_raw = extract_sql(result)
final_sql = fix_sql_output(result)

# ---------------------------
# STEP 5: PRINT
# ---------------------------
print("\nCLEAN RAW OUTPUT:\n")
print(cleaned_raw)

print("\nFINAL SQL:\n")
print(final_sql)


CLEAN RAW OUTPUT:

CREATE TYPE data AS ENUM ('a', 'b', 'c');

FINAL SQL:

CREATE TABLE tasks (
  id SERIAL PRIMARY KEY,
  status TEXT
);


In [30]:
FINAL_DIR = "/kaggle/working/final_adapter"

model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)

print(f"✅ Training Complete. Adapter saved to: {FINAL_DIR}")
!ls -lh {FINAL_DIR}

✅ Training Complete. Adapter saved to: /kaggle/working/final_adapter
total 1.2G
-rw-r--r-- 1 root root 1.5K Mar 30 13:20 adapter_config.json
-rw-r--r-- 1 root root 1.2G Mar 30 13:20 adapter_model.safetensors
-rw-r--r-- 1 root root 1.5K Mar 30 13:20 chat_template.jinja
-rw-r--r-- 1 root root  560 Mar 30 13:20 processor_config.json
-rw-r--r-- 1 root root 5.2K Mar 30 13:20 README.md
-rw-r--r-- 1 root root  725 Mar 30 13:20 tokenizer_config.json
-rw-r--r-- 1 root root  32M Mar 30 13:20 tokenizer.json


In [37]:
from unsloth import FastVisionModel

FastVisionModel.for_inference(model)

inputs = processor(
    text="### Instruction:\nGenerate PostgreSQL schema from this UI\n\n### Output:",
    images=None, 
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100)

print("🧪 Test Output:\n", processor.decode(outputs[0], skip_special_tokens=True))

🧪 Test Output:
 ### Instruction:
Generate PostgreSQL schema from this UI

### Output:
CREATE TYPE data_filter AS ENUM ('all', 'active', 'inactive', 'deleted');
CREATE TYPE order_field AS ENUM ('created_at', 'updated_at', 'metadata', 'content');
CREATE TABLE records (
  id INT PRIMARY KEY AUTO_INCREMENT,
  type VARCHAR(100),
  content JSON,
  is_active BOOLEAN DEFAULT TRUE,
  metadata JSON,
  created_by INT,
  updated
